In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1. How many lesson pages
len(documents)

72

In [4]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [5]:
# Q2. Indexing and searching
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [6]:
index.search('How does the agentic loop keep calling the model until it stops?')[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
# Q3. RAG
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase

from openai import OpenAI
openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [17]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('It keeps calling the model in a `while True` loop and stops only when the model returns a response with no `function_call` items.\n\nThe code checks each response:\n- if there is a function call, it runs the tool, appends the tool result to `messages`, and loops again\n- if there are no function calls, it `break`s out of the loop\n\nSo the exit condition is: **no function calls this turn**.',
 ResponseUsage(input_tokens=7135, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=96, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7231))

In [9]:
# Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

In [15]:
# Q5. RAG with chunking

index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)


In [16]:
assistant_chunked = RAGBase(
    index=index_chunks,
    llm_client=openai_client,
)

In [18]:
assistant_chunked.rag('How does the agentic loop keep calling the model until it stops?')

('It keeps calling the model inside a `while True` loop. After each response, the code checks whether any item is a `function_call`:\n\n- if there is a function call, it runs the tool, appends the tool output to `messages`, and continues looping\n- if there are no function calls in that turn, it breaks out of the loop\n\nSo the stop condition is: **no function calls this turn means the model has given its final answer, and the loop ends.**',
 ResponseUsage(input_tokens=2318, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2423))

In [25]:
# Q6. Turning it into an agent
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [29]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lessons for entries matching the given query.
    """
    return index_chunks.search(
        query,
        num_results=5
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [34]:
result.cost

CostInfo(input_cost=Decimal('0.00800775'), output_cost=Decimal('0.0022545'), total_cost=Decimal('0.01026225'))